## **Upload The All Dataset From Google maps**

In [ ]:
import zipfile
import os

ZIP_PATH = "/content/Cleaned Data From Google Maps.zip"
EXTRACT_PATH = "/content/extracted_google_maps_data"

with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("Extraction done!")

Extraction done!


In [ ]:
import os
import shutil
import pandas as pd

# paths
INPUT_XLSX_FOLDER = "/content/extracted_google_maps_data/Cleaned Data From Google Maps"
OUTPUT_CSV_FOLDER = "/content/Google_Maps_CSV_FIXED1"

# remove old output folder if exists
if os.path.exists(OUTPUT_CSV_FOLDER):
    shutil.rmtree(OUTPUT_CSV_FOLDER)

os.makedirs(OUTPUT_CSV_FOLDER, exist_ok=True)

# collect all xlsx files
xlsx_files = []

for root, dirs, files in os.walk(INPUT_XLSX_FOLDER):
    for file in files:
        if file.lower().endswith(".xlsx"):
            xlsx_files.append(os.path.join(root, file))

print("Number of XLSX files:", len(xlsx_files))

# first pass: collect all column names in one unified order
all_columns = []

for path in xlsx_files:
    try:
        temp_df = pd.read_excel(path, nrows=0)

        # unify stars column name
        temp_df.columns = [str(c).strip() for c in temp_df.columns]

        for c in temp_df.columns:
            if c.lower() == "stars":
                c = "Stars"

            if c not in all_columns:
                all_columns.append(c)

    except Exception as e:
        print("Error reading columns:", path, e)

print("Total unified columns:", len(all_columns))

# second pass: convert xlsx to csv with same column order
converted_count = 0
error_count = 0

for src_path in xlsx_files:
    try:
        df_excel = pd.read_excel(src_path, dtype=str)

        # clean column names
        df_excel.columns = [str(c).strip() for c in df_excel.columns]

        # unify stars column name
        rename_map = {}
        for c in df_excel.columns:
            if c.lower() == "stars":
                rename_map[c] = "Stars"

        df_excel = df_excel.rename(columns=rename_map)

        # reorder columns using the unified schema
        df_excel = df_excel.reindex(columns=all_columns)

        # keep same folder structure
        relative_path = os.path.relpath(os.path.dirname(src_path), INPUT_XLSX_FOLDER)
        dest_folder = os.path.join(OUTPUT_CSV_FOLDER, relative_path)
        os.makedirs(dest_folder, exist_ok=True)

        csv_name = os.path.splitext(os.path.basename(src_path))[0] + ".csv"
        dest_path = os.path.join(dest_folder, csv_name)

        # save csv
        df_excel.to_csv(dest_path, index=False, encoding="utf-8-sig")

        converted_count += 1

    except Exception as e:
        error_count += 1
        print("Error converting:", src_path, e)

print("Converted files:", converted_count)
print("Errors:", error_count)
print("Output folder:", OUTPUT_CSV_FOLDER)

Number of XLSX files: 442
Total unified columns: 79
Converted files: 442
Errors: 0
Output folder: /content/Google_Maps_CSV_FIXED1


## **Data Ingestion (PySpark)**

In [ ]:
import os
from glob import glob
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Read Fixed Google Maps CSV Files") \
    .getOrCreate()

DATA_PATH = "/content/Google_Maps_CSV_FIXED1"

csv_files = glob(DATA_PATH + "/**/*.csv", recursive=True)

print("Number of CSV files:", len(csv_files))

df_raw = spark.read \
    .option("header", True) \
    .option("inferSchema", False) \
    .option("recursiveFileLookup", "true") \
    .option("multiLine", True) \
    .option("escape", '"') \
    .option("quote", '"') \
    .option("mode", "PERMISSIVE") \
    .csv(DATA_PATH)

print("Total rows:", df_raw.count())
print("Total columns:", len(df_raw.columns))

df_raw.select("Stars").show(20, truncate=False)

Number of CSV files: 442
Total rows: 1016595
Total columns: 79
+-----+
|Stars|
+-----+
|5    |
|5    |
|5    |
|5    |
|5    |
|5    |
|5    |
|3    |
|5    |
|5    |
|5    |
|3    |
|5    |
|5    |
|5    |
|5    |
|5    |
|5    |
|5    |
|5    |
+-----+
only showing top 20 rows


## **Data preparation**

In [ ]:
from pyspark.sql.functions import col, trim, when

df_labeled = df_raw.withColumn("Stars_clean", trim(col("Stars").cast("string")))

df_labeled = df_labeled.withColumn(
    "Sentiment",
    when((col("Stars_clean") == "1") | (col("Stars_clean") == "2"), "negative")
    .when(col("Stars_clean") == "3", "neutral")
    .when((col("Stars_clean") == "4") | (col("Stars_clean") == "5"), "positive")
)

df_labeled.groupBy("Stars_clean").count().orderBy("Stars_clean").show()
df_labeled.groupBy("Sentiment").count().show()

print("Total rows after labeling:", df_labeled.count())

+-----------+------+
|Stars_clean| count|
+-----------+------+
|       NULL|   380|
|          1| 65871|
|          2| 34734|
|          3|103035|
|          4|166508|
|          5|646067|
+-----------+------+

+---------+------+
|Sentiment| count|
+---------+------+
| positive|812575|
|     NULL|   380|
|  neutral|103035|
| negative|100605|
+---------+------+

Total rows after labeling: 1016595


## **Cleaned Data Storage**

In [ ]:
# define storage path
STORAGE_PATH = "/content/google_maps_parquet"

# save as parquet
df_labeled.write \
    .mode("overwrite") \
    .parquet(STORAGE_PATH)

print("Parquet storage created ")

Parquet storage created 


In [ ]:
df_stored = spark.read.parquet("/content/google_maps_parquet")

print("Stored rows:", df_stored.count())
print("Stored columns:", len(df_stored.columns))

df_stored.show(5, truncate=False)

Stored rows: 1016595
Stored columns: 81
+-------------+-------------+-------------+-------------+----+-----+-----+----------------------------------------------------------------------------------------------------+----------+-------+------+----------------------------------------------------------------------------------------------------+----------+-----------+---------------+---------------+-----------+---------------+----------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------+-----------+-----------+----------------------------------------------------------------------------------------------------+--------------+------+--------------+--------------+--------------+------------------------+----------------+------------------------+-------+--

## **Data Processing (Apache Spark)**

In [ ]:
from pyspark.sql.functions import col, coalesce, length, year, month, to_timestamp

# create processing dataframe
df_processing = df_stored

# standardize text column
df_processing = df_processing.withColumn(
    "Review_Text",
    coalesce(col("Text_TR"), col("Text7"), col("Text_Orig"), col("text23"))
)

# standardize date column
df_processing = df_processing.withColumn(
    "Review_Date",
    coalesce(col("Date"), col("publishedAtDate"))
)

# convert date to timestamp
df_processing = df_processing.withColumn(
    "Review_Timestamp",
    to_timestamp(col("Review_Date"))
)

# create review length feature
df_processing = df_processing.withColumn(
    "Review_Length",
    length(col("Review_Text"))
)

# extract year and month
df_processing = df_processing.withColumn("Year", year(col("Review_Timestamp")))
df_processing = df_processing.withColumn("Month", month(col("Review_Timestamp")))

# final processing dataset without deleting rows
df_final = df_processing.select(
    "Review_Text",
    "Stars",
    "Sentiment",
    "Review_Length",
    "Year",
    "Month"
)

# check final data
print("Rows:", df_final.count())
print("Columns:", len(df_final.columns))

df_final.printSchema()
df_final.show(5, truncate=False)

Rows: 1016595
Columns: 6
root
 |-- Review_Text: string (nullable = true)
 |-- Stars: string (nullable = true)
 |-- Sentiment: string (nullable = true)
 |-- Review_Length: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)

+----------------------------------------------------------------------------------------------------+-----+---------+-------------+----+-----+
|Review_Text                                                                                         |Stars|Sentiment|Review_Length|Year|Month|
+----------------------------------------------------------------------------------------------------+-----+---------+-------------+----+-----+
|تم تطويره عن السابق و الشيء المميز لا يوجد قرود المزعجة ممتاز للزيارة والاستمتاع بقمم الجبال الشاهقة|5    |positive |100          |2025|10   |
|NULL                                                                                                |5    |positive |NULL         |2025|10   |
|الاول 

In [ ]:
import shutil

shutil.make_archive("/content/google_maps_parquet", 'zip', "/content/google_maps_parquet")

print("Zipped successfully!")

Zipped successfully!


In [ ]:
# install required libraries
!pip install -q transformers datasets evaluate accelerate

# imports
import os
import numpy as np
import pandas as pd
import torch

from pyspark.sql.functions import col, coalesce, when
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

# read parquet data
df_stored = spark.read.parquet("/content/google_maps_parquet")

# prepare data for CamelBERT
df_model = df_stored.select(
    coalesce(col("Text_TR"), col("Text7"), col("Text_Orig"), col("text23")).alias("text"),
    col("Sentiment")
)

# keep only rows with text
df_model = df_model.filter(col("text").isNotNull())
df_model = df_model.filter(col("text") != "")

# remove neutral class for binary classification
df_model = df_model.filter(col("Sentiment") != "neutral")

# create numeric label
df_model = df_model.withColumn(
    "label",
    when(col("Sentiment") == "negative", 0)
    .when(col("Sentiment") == "positive", 1)
)

# keep final columns
df_model = df_model.select("text", "label")

print("Spark rows:", df_model.count())
df_model.groupBy("label").count().show()

# convert to pandas
df5 = df_model.toPandas()

# clean final pandas dataframe
df5 = df5.dropna(subset=["text", "label"])
df5["text"] = df5["text"].astype(str)
df5["label"] = df5["label"].astype(int)

print("Full dataset shape:", df5.shape)
print(df5["label"].value_counts())

# create Hugging Face dataset
dataset = Dataset.from_pandas(df5)

# split dataset
dataset = dataset.train_test_split(
    test_size=0.2,
    seed=42,
    stratify_by_column="label"
)

train_dataset = dataset["train"]
test_dataset = dataset["test"]

print(train_dataset)
print(test_dataset)

# load CamelBERT tokenizer
MODEL_NAME = "CAMeL-Lab/bert-base-arabic-camelbert-da"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# tokenize function
def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

# tokenize datasets
train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# remove text column after tokenization
train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])

# set torch format
train_dataset.set_format("torch")
test_dataset.set_format("torch")

# load CamelBERT model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

# define metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    acc = accuracy_score(labels, predictions)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

# training arguments
training_args = TrainingArguments(
    output_dir="/content/camelbert_results_full",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="/content/camelbert_logs_full",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=2,
    report_to="none"
)

# create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# train model on full dataset
trainer.train()

# evaluate model
results = trainer.evaluate()
print(results)

# predictions
predictions_output = trainer.predict(test_dataset)

y_pred = np.argmax(predictions_output.predictions, axis=1)
y_true = predictions_output.label_ids

print("Classification Report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=["negative", "positive"],
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

# save model and tokenizer
SAVE_PATH = "/content/camelbert_google_maps_full_model"

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("Model saved successfully ✅")

# prediction function
def predict_sentiment(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    pred = torch.argmax(outputs.logits, dim=1).item()

    return "positive" if pred == 1 else "negative"

# test prediction
print(predict_sentiment("المكان جميل والخدمة ممتازة"))